In [1]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier,XGBModel
import xgboost as xgb
from xgboost import plot_importance
import matplotlib.pyplot as plt
from transformers import AutoModel


Matplotlib is building the font cache; this may take a moment.


In [ ]:
from trl import DPOTrainer,PPOTrainer

In [2]:
import torch
from torch import nn
from torch.nn import functional as F
from peft import PeftModel
from torch.autograd import Variable

In [3]:
tensor_  = torch.FloatTensor([[1,2],[3,4]])
variable_ = Variable(tensor_,requires_grad=True)
print(tensor_)
print(variable_)
print(tensor_*tensor_)
t_out = torch.mean(tensor_*tensor_)
v_out = torch.mean(variable_*variable_)
print(t_out)
print(v_out)
v_out.backward()
print(variable_.grad)

tensor([[1., 2.],
        [3., 4.]])
tensor([[1., 2.],
        [3., 4.]], requires_grad=True)
tensor([[ 1.,  4.],
        [ 9., 16.]])
tensor(7.5000)
tensor(7.5000, grad_fn=<MeanBackward0>)
tensor([[0.5000, 1.0000],
        [1.5000, 2.0000]])


In [ ]:
# FFN with switchGLU激活函数
hid_dim = 10
inter_dim = 5
act_fn = nn.SiLU()
gate_proj = nn.Linear(hid_dim,inter_dim)
up_proj = nn.Linear(hid_dim,inter_dim,bias=False)
down_proj = nn.Linear(inter_dim,hid_dim,bias=False)
batch_size = 3
input_ = torch.randn(batch_size,hid_dim)
output = down_proj(act_fn(gate_proj(input_))*up_proj(input_))
print(output)

tensor([[-0.0936, -0.0428, -0.0495,  0.0092, -0.1267,  0.0188, -0.1084,  0.1142,
          0.0237,  0.0294],
        [-0.0558, -0.1481, -0.1444,  0.1868, -0.0977, -0.0635, -0.2085,  0.0736,
          0.2574,  0.0994],
        [-0.1128,  0.0055, -0.0109, -0.1441, -0.1426,  0.0924, -0.1320,  0.1326,
         -0.1203,  0.0115]], grad_fn=<MmBackward0>)


In [ ]:
import torch
import torch.nn.functional as F

# ======================== 改变维度操作 ========================
# squeeze: 移除大小为1的维度
x = torch.randn(1, 3, 1, 5)
print("squeeze示例：", x.squeeze().shape)  # 移除所有大小为1的维度

# unsqueeze: 增加大小为1的维度
x = torch.randn(3, 5)
print("unsqueeze示例：", x.unsqueeze(0).shape)  # 在第0维增加

# permute: 重新排列维度
x = torch.randn(2, 3, 4)
print("permute示例：", x.permute(1, 2, 0).shape)  # 改变维度顺序

# reshape: 调整张量形状
x = torch.randn(2, 3, 4)
print("reshape示例：", x.reshape(6, 4).shape)  # 调整为6行4列

# ======================== 拼接与分割操作 ========================
# cat: 按维度拼接
x = torch.randn(2, 3)
y = torch.randn(2, 3)
print("cat示例：", torch.cat([x, y], dim=0).shape)  # 按行拼接

# stack: 按新维度堆叠
x = torch.randn(2, 3)
y = torch.randn(2, 3)
print("stack示例：", torch.stack([x, y], dim=0).shape)  # 在第0维堆叠

# chunk: 分割为固定块数
x = torch.randn(6, 4)
chunks = torch.chunk(x, 3, dim=0)  # 按第0维分成3块
print("chunk示例：", [chunk.shape for chunk in chunks])

# split: 按块大小分割
splits = torch.split(x, [2, 4], dim=0)  # 按大小[2,4]分割
print("split示例：", [split.shape for split in splits])

# ======================== 基本数学操作 ========================
# add, mul: 加法、乘法
x = torch.tensor([1, 2, 3])
y = torch.tensor([4, 5, 6])
print("add示例：", torch.add(x, y))  # 元素相加
print("mul示例：", torch.mul(x, y))  # 元素相乘

# matmul: 矩阵乘法
x = torch.randn(2, 3)
y = torch.randn(3, 4)
print("matmul示例：", torch.matmul(x, y).shape)  # 矩阵乘法

# sum, mean: 求和、求均值
x = torch.tensor([[1, 2], [3, 4]])
print("sum示例：", torch.sum(x))  # 总和
print("mean示例：", torch.mean(x.float(), dim=1))  # 每行均值

# ======================== 逻辑操作 ========================
# eq, gt: 等于、大于
x = torch.tensor([1, 2, 3])
y = torch.tensor([2, 2, 4])
print("eq示例：", torch.eq(x, y))  # 元素是否相等
print("gt示例：", torch.gt(x, y))  # 元素是否大于

# argmax: 获取最大值索引
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("argmax示例：", torch.argmax(x, dim=1))  # 每行最大值索引

# ======================== Dropout操作 ========================
# nn.functional.dropout
x = torch.randn(5, 5)
output_train = F.dropout(x, p=0.5, training=True)  # 训练模式
output_eval = F.dropout(x, p=0.5, training=False)  # 评估模式
print("Dropout训练模式：", output_train)
print("Dropout评估模式：", output_eval)


In [2]:
xgb1 = XGBClassifier(
                    alpha=1,  # L1正则化系数，默认为1
                    seed=4,  # 随机种子 复现
                    scale_pos_weight=1,  # 正样本的权重，在二分类任务中，当正负样本比例失衡时，设置正样本的权重，模型效果更好。例如，当正负样本比例为1:10时scale_pos_weight=10。
                    num_class=2,
                    nthread=-1,  # nthread=-1时，使用全部CPU进行并行运算（默认）nthread=1时，使用1个CPU进行运算。
                    silent=1,  # silent=0时，不输出中间过程（默认）silent=1时，输出中间过程

                    subsample=0.8,  # 使用的数据占全部训练集的比例。防止overfitting。默认值为1，典型值为0.5-1。
                    colsample_bytree=0.8,  # 使用的特征占全部特征的比例。防止overfitting。默认值为1，典型值为0.5-1。
                    colsample_bylevel=0.7,

                    learning_rate=0.01,  # 学习率，控制每次迭代更新权重时的步长，值越小，训练越慢。默认0.3，典型值为0.01-0.2。
                    n_estimators=1000000,  # 总共迭代的次数，即决策树的个数，数值大没关系，cv会自动返回合适的n_estimators
                    max_depth=5,  # 树的深度，默认值为6，典型值3-10。
                    min_child_weight=2,  # 值越大，越容易欠拟合；值越小，越容易过拟合（值较大时，避免模型学习到局部的特殊样本）。默认值为1
                    gamma=0,  # 惩罚项系数，指定节点分裂所需的最小损失函数下降值。
                    objective='multi:softprob',
                    )

In [ ]:
xgb1.load_model()

In [5]:
xgb1.min_child_weight

2

In [ ]:
xgb.DMatrix(feature_names=)

In [7]:
bst = xgb.Booster(model_file="../Modules_test/XGBoost/xgboost_classification/model/xgb.model")

/root/miniconda3/envs/wck/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [13:55:16] WARNING: /workspace/src/learner.cc:1071: Loading model from XGBoost < 1.0.0, consider saving it again for improved compatibility
  warnings.warn(smsg, UserWarning)


In [21]:
pra_fscore = bst.get_fscore()
pra_score = bst.get_score()

In [20]:
print(pra_fscore)

{'f0': 114.0, 'f1': 77.0, 'f2': 40.0, 'f3': 166.0, 'f4': 43.0, 'f5': 46.0, 'f6': 171.0, 'f7': 71.0, 'f8': 82.0, 'f9': 27.0, 'f10': 44.0, 'f11': 60.0, 'f12': 58.0, 'f13': 77.0, 'f14': 85.0, 'f15': 107.0, 'f16': 32.0, 'f17': 42.0, 'f18': 36.0, 'f19': 48.0, 'f20': 38.0, 'f21': 45.0, 'f22': 53.0, 'f23': 23.0, 'f24': 24.0, 'f25': 45.0, 'f26': 40.0, 'f27': 37.0, 'f28': 26.0, 'f29': 45.0, 'f30': 106.0, 'f31': 24.0, 'f32': 40.0, 'f33': 78.0, 'f34': 27.0, 'f35': 71.0, 'f36': 36.0, 'f37': 65.0, 'f38': 59.0, 'f39': 54.0, 'f40': 61.0, 'f42': 40.0, 'f43': 45.0, 'f44': 84.0, 'f45': 93.0, 'f46': 31.0, 'f47': 54.0, 'f48': 54.0, 'f49': 91.0, 'f51': 53.0, 'f53': 40.0, 'f54': 60.0, 'f55': 2.0, 'f57': 40.0, 'f60': 80.0, 'f61': 2.0, 'f62': 4.0, 'f63': 76.0, 'f65': 34.0, 'f66': 52.0, 'f71': 21.0, 'f73': 50.0, 'f76': 34.0, 'f77': 42.0, 'f78': 55.0, 'f80': 30.0, 'f81': 72.0, 'f82': 15.0, 'f83': 64.0, 'f84': 91.0, 'f86': 40.0, 'f88': 34.0, 'f89': 24.0, 'f90': 27.0, 'f91': 58.0, 'f92': 48.0, 'f93': 49.0, 'f94':

In [22]:
print(pra_score)

{'f0': 114.0, 'f1': 77.0, 'f2': 40.0, 'f3': 166.0, 'f4': 43.0, 'f5': 46.0, 'f6': 171.0, 'f7': 71.0, 'f8': 82.0, 'f9': 27.0, 'f10': 44.0, 'f11': 60.0, 'f12': 58.0, 'f13': 77.0, 'f14': 85.0, 'f15': 107.0, 'f16': 32.0, 'f17': 42.0, 'f18': 36.0, 'f19': 48.0, 'f20': 38.0, 'f21': 45.0, 'f22': 53.0, 'f23': 23.0, 'f24': 24.0, 'f25': 45.0, 'f26': 40.0, 'f27': 37.0, 'f28': 26.0, 'f29': 45.0, 'f30': 106.0, 'f31': 24.0, 'f32': 40.0, 'f33': 78.0, 'f34': 27.0, 'f35': 71.0, 'f36': 36.0, 'f37': 65.0, 'f38': 59.0, 'f39': 54.0, 'f40': 61.0, 'f42': 40.0, 'f43': 45.0, 'f44': 84.0, 'f45': 93.0, 'f46': 31.0, 'f47': 54.0, 'f48': 54.0, 'f49': 91.0, 'f51': 53.0, 'f53': 40.0, 'f54': 60.0, 'f55': 2.0, 'f57': 40.0, 'f60': 80.0, 'f61': 2.0, 'f62': 4.0, 'f63': 76.0, 'f65': 34.0, 'f66': 52.0, 'f71': 21.0, 'f73': 50.0, 'f76': 34.0, 'f77': 42.0, 'f78': 55.0, 'f80': 30.0, 'f81': 72.0, 'f82': 15.0, 'f83': 64.0, 'f84': 91.0, 'f86': 40.0, 'f88': 34.0, 'f89': 24.0, 'f90': 27.0, 'f91': 58.0, 'f92': 48.0, 'f93': 49.0, 'f94':

In [16]:
# plot_importance(bst)
# plt.show()


In [8]:
dump_model = bst.get_dump()

In [12]:
bst.dump_model("./xgb_tree.json")

In [11]:
print(len(dump_model))

482


In [3]:
xgb1.get_xgb_params()

{'objective': 'multi:softprob',
 'base_score': None,
 'booster': None,
 'colsample_bylevel': 0.7,
 'colsample_bynode': None,
 'colsample_bytree': 0.8,
 'device': None,
 'eval_metric': None,
 'gamma': 0,
 'grow_policy': None,
 'interaction_constraints': None,
 'learning_rate': 0.01,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': 5,
 'max_leaves': None,
 'min_child_weight': 2,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': None,
 'reg_alpha': None,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': 1,
 'subsample': 0.8,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None,
 'alpha': 1,
 'seed': 4,
 'num_class': 2,
 'nthread': -1,
 'silent': 1}

In [14]:
iris = load_iris()

In [25]:
x = iris.data
y = (iris.target != 0) * 1
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 创建逻辑回归模型
log_reg = LogisticRegression()

lin_reg = LinearRegression()

# 训练模型
log_reg.fit(X_train, y_train)

lin_reg.fit(X_train,y_train)

# 预测
y_pred = log_reg.predict(X_test)

In [22]:
y_proba = log_reg.predict_proba(X_test)

In [26]:
y_pred_lin = lin_reg.predict(X_test)

In [27]:
y_pred_lin[:5]

array([ 0.93141246, -0.00867146,  1.44710026,  0.88479558,  0.93261692])

In [23]:
y_proba[:5]

array([[4.55385779e-03, 9.95446142e-01],
       [9.58520512e-01, 4.14794882e-02],
       [5.40034424e-06, 9.99994600e-01],
       [6.13513643e-03, 9.93864864e-01],
       [2.25748581e-03, 9.97742514e-01]])

In [20]:
y_pred[:5]

array([1, 0, 1, 1, 1])

In [21]:
y_test[:5]

array([1, 0, 1, 1, 1])

In [11]:
x[:4]

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2]])

In [12]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [1]:
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

In [2]:
API_BASE = "https://api.lingyiwanwu.com/v1"
API_KEY = "d45ad0328e0847dfab5265178416f0b9"
client_01 = OpenAI(
    api_key=API_KEY,
    base_url=API_BASE
)

def pro_api_01(message):
    completion = client_01.chat.completions.create(
        model="yi-large",
        messages=[{"role": "user", "content": message}]
    )
    output = completion.choices[0].message.content
    return output

In [ ]:
prompt_str_list = [
    "中国的首都是哪里",
    "日本的首都是哪里",
    "韩国的首都是哪里",
    "印度的首都是哪里",
    "俄罗斯的首都是哪里"
]

with ThreadPoolExecutor(max_workers=len(prompt_str_list)) as executor:
    results = list(executor.map(pro_api_01, prompt_str_list))

In [5]:
import asyncio
import aiohttp

# 定义一个异步函数
async def fetch_data(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            return await response.text()

# 定义一个异步的主函数
async def main():
    url = 'http://httpbin.org/get'  # 这是一个用于测试HTTP请求的公共API
    html = await fetch_data(url)  # 等待异步请求完成
    print(html[:200])  # 打印获取到的部分HTML内容

# 获取当前事件循环
loop = asyncio.get_event_loop()

# 如果事件循环正在运行，使用 ensure_future 来安排协程的执行
if loop.is_running():
    task = loop.create_task(main())
else:
    # 否则，直接运行main函数
    print("else")
    loop.run_until_complete(main())

{
  "args": {}, 
  "headers": {
    "Accept": "*/*", 
    "Accept-Encoding": "gzip, deflate, br", 
    "Host": "httpbin.org", 
    "User-Agent": "Python/3.12 aiohttp/3.10.5", 
    "X-Amzn-Trace-Id": "


In [4]:
20*0.25

5.0